In [1]:
import pandas as pd

In [2]:
import yfinance as yf
import datetime as dt

In [3]:
# Define your date range
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)

stk_data = yf.download('TATACONSUM.NS', start=start, end=end)

[*********************100%***********************]  1 of 1 completed


In [4]:
stk_data=stk_data[["Open","High","Low","Close"]]
#stk_data.to_csv("Tatacoffee13_21.csv")

In [5]:
#column="Close"

In [6]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2144, 4)


In [7]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [8]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1715
X_train length: (1715, 4)
X_test length: (429, 4)
y_train length: (1715, 4)
y_test length: (429, 4)


In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [23]:

from statsmodels.tsa.api import VAR
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error

def cominbation(dataset, listt):
    print(listt)
    datasetTwo = dataset[listt]
    test_obs = 28
    train = datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    
    # Initialize the model ONCE here
    model = VAR(train)
    
    # Loop only through the fitting process
    for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
        
    # Select order and fit the final model
    x = model.select_order(maxlags=12)
    order = x.selected_orders["aic"]
    result = model.fit(order)
    
    # Forecast
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values, steps=28) 
    
    # Keep historical dates alignment for the CSV export
    preds = pd.DataFrame(pred, columns=listt, index=test.index)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    
    # Metrics calculations
    rmse = round(root_mean_squared_error(test, pred))
    mape = mean_absolute_percentage_error(test, pred)
    
    # Append to performance tracking
    performance["Model"].append(str(listt)) # string conversion prevents future dataframe nesting errors
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    
    perf = pd.DataFrame(performance)
    return perf, result, pred

In [24]:
listt=["Close","High","Open","Low"]
#listt=["AQI_calculated","PM10","PM2.5","NOx","NO2","NO","NH3","SO2","CO",'year']


In [25]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']
Order = 1
AIC:  -44.50374219541911
BIC:  -44.45024990140983

Order = 2
AIC:  -44.576674248507715
BIC:  -44.480350625995186

Order = 3
AIC:  -44.59354680607006
BIC:  -44.45435848192617

Order = 4
AIC:  -44.612136842005526
BIC:  -44.43005040078953

Order = 5
AIC:  -44.63326392639627
BIC:  -44.40824591028103

Order = 6
AIC:  -44.65075301422418
BIC:  -44.38276992292228

Order = 7
AIC:  -44.64990980019501
BIC:  -44.33892809088466

Order = 8
AIC:  -44.662347107696185
BIC:  -44.308333194947

Order = 9
AIC:  -44.66280214116603
BIC:  -44.265722396864646

Order = 10
AIC:  -44.66785984721541
BIC:  -44.22768060049091



In [26]:
data1

,Open,High,Low,Close
0,0.042448,0.043230,0.044432,0.044287
1,0.044571,0.047463,0.047579,0.047482
2,0.047699,0.047128,0.049827,0.047818
3,0.046694,0.045959,0.047916,0.047146
4,0.045912,0.048298,0.049490,0.046585
...,...,...,...,...
2139,0.817668,0.820204,0.802074,0.811609
2140,0.802473,0.807835,0.776410,0.774769
2141,0.772211,0.774072,0.762781,0.764987
2142,0.773351,0.772810,0.769404,0.768925


In [27]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"['Close', 'High', 'Open', 'Low']",0,0.033589,11,28
1,"['Close', 'High', 'Open', 'Low']",0,0.033589,11,28
